# Open-Meteo Extraction Notebook

Note on Elevation Data Extraction:
The Open-Meteo API was used to fetch the public elevation data (DEM) for the unique coordinates. Because the Snowflake environment blocks external HTTP requests (NameResolutionError due to security policies), this specific extraction block was executed locally. The resulting data was saved as elevation_data.csv and uploaded to the Snowflake environment to be merged with the main dataset. The original extraction code is preserved below for transparency and reproducibility.

## Environment preparation

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df

In [ ]:
# If you want to reproduce this code, set this variable to 'True'
RUN_API = False

In [ ]:
water_quality_df = (
    pd.read_csv('/content/water_quality_training_dataset.csv')
    [['Latitude', 'Longitude']]
)

validation_df = (
    pd.read_csv('/content/submission_template.csv')
    [['Latitude', 'Longitude']]
)

unique_coords_df = pd.concat(
    [water_quality_df, validation_df],
    ignore_index=True
).drop_duplicates()

In [ ]:
if RUN_API:
    elevations = []
    batch_size = 100
    
    # Sending batch requests
    for i in range(0, len(unique_coords_df), batch_size):
        batch = unique_coords_df.iloc[i:i+batch_size]
        
        # Join each line with ',' 
        lats = ",".join(batch['Latitude'].astype(str))
        lons = ",".join(batch['Longitude'].astype(str))
        
        # Calls API
        url = f"https://api.open-meteo.com/v1/elevation?latitude={lats}&longitude={lons}"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                elevations.extend(data['elevation'])
            else:
                print(f"⚠️ Error on batch {i}. Filling with NaN.")
                elevations.extend([np.nan] * len(batch))
        except Exception as e:
            print(f"⚠️ Connection failed: {e}")
            elevations.extend([np.nan] * len(batch))
            
        # Waits for another request
        time.sleep(1) 
        
    # Adds new column
    unique_coords_df['Elevation'] = elevations
    
    # Saves dataframe
    save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/intermediate/'
    save_df(unique_coords_df, 'elevation_feature.csv', save_path)